# 8. Fourier basis and Fourier matrix  [6]

TOML-MIRI — Linear Algebra for Data Science, exploration exercises.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import circulant
from threadpoolctl import threadpool_limits
from ipywidgets import interact, IntSlider, FloatSlider, Dropdown

np.set_printoptions(precision=4, suppress=True)

def check(label, ok):
    print(f"{'✓' if ok else '✗'}  {label}")

---
## 8.1 Look at the Fourier matrix

**Task.** Build $F_N$ for $N = 8$ and $N = 64$ from its definition. Plot the real and imaginary parts as heatmaps, and plot each of the first 8 columns as a sampled cosine/sine.

$$\omega_N = e^{j2\pi/N}, \qquad (F_N)_{ki} = \frac{1}{\sqrt{N}}\,\omega_N^{-ki}, \qquad k, i = 0, \dots, N-1.$$

### Prediction (write this *before* running)

- What pattern will the heatmap of $\mathrm{Re}\,F_N$ show?
- Which column oscillates fastest?

*(fill in)*

In [ ]:
def fourier_matrix(N):
    omega = np.exp(2j * np.pi / N)
    k = np.arange(N)
    return omega ** (-np.outer(k, k)) / np.sqrt(N)

fig, axes = plt.subplots(2, 2, figsize=(11, 10))
for col, N in enumerate([8, 64]):
    F = fourier_matrix(N)
    for row, (part, name) in enumerate([(F.real, "Re"), (F.imag, "Im")]):
        im = axes[row, col].imshow(part * np.sqrt(N), cmap="RdBu_r", vmin=-1, vmax=1)
        axes[row, col].set_title(f"{name} √N·F_{N}")
        axes[row, col].set_xlabel("column i"); axes[row, col].set_ylabel("row k")
fig.colorbar(im, ax=axes, shrink=0.6)
plt.show()

In [ ]:
N = 64
F = fourier_matrix(N)
i = np.arange(N)
tt = np.linspace(0, N, 600)

def plot_columns(cols):
    fig, axes = plt.subplots(2, 4, figsize=(19, 6.5), sharey=True)
    for k, ax in zip(cols, axes.ravel()):
        col = F[:, k] * np.sqrt(N)
        ax.plot(tt, np.cos(2 * np.pi * k * tt / N), color="C0", alpha=0.25)
        ax.plot(tt, -np.sin(2 * np.pi * k * tt / N), color="C3", alpha=0.25)
        ax.plot(i, col.real, "o", color="C0", ms=3, label="Re (cos)")
        ax.plot(i, col.imag, "o", color="C3", ms=3, label="Im (−sin)")
        ax.set_title(f"column k = {k}"); ax.grid(alpha=0.3)
    axes[0, 0].legend(loc="lower left")
    plt.show()

print(f"first 8 columns of F_{N}")
plot_columns(range(8))
print("columns near N/2 and the last one")
plot_columns([N // 2 - 2, N // 2 - 1, N // 2, N // 2 + 1, N // 2 + 2, N - 3, N - 2, N - 1])

### Verify numerically

In [ ]:
F4_notes = 0.5 * np.array([[1, 1, 1, 1], [1, -1j, -1, 1j], [1, -1, 1, -1], [1, 1j, -1, -1j]])
check("F_4 matches (6.3.2)", np.allclose(fourier_matrix(4), F4_notes))
for N in [8, 64]:
    F = fourier_matrix(N)
    check(f"F_{N}ᴴ F_{N} = I  (unitary)", np.allclose(F.conj().T @ F, np.eye(N)))
    check(f"F_{N}ᵀ F_{N} ≠ I", not np.allclose(F.T @ F, np.eye(N)))
    check(f"F_{N} is symmetric (F = Fᵀ)", np.allclose(F, F.T))
    check(f"column N−k is the complex conjugate of column k", np.allclose(F[:, 1:], F[:, :0:-1].conj()))

In [ ]:
F = fourier_matrix(8)
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
axes[0].imshow(np.abs(F.conj().T @ F), cmap="Greys"); axes[0].set_title("|F₈ᴴ F₈|")
axes[1].imshow(np.abs(F.T @ F), cmap="Greys"); axes[1].set_title("|F₈ᵀ F₈|")
plt.show()

---
### Explain (hand-in — in your own words)

**1. Verify $F_N^H F_N = I$. Why must you use the conjugate transpose and not the plain transpose?**
*(Hint: the length of a complex number is $|z|^2 = \bar z z$, not $z^2$. What is $\sum_i \omega^{-ki}\omega^{-ki}$ compared with $\sum_i \overline{\omega^{-ki}}\,\omega^{-ki}$? The heatmap of $|F^\top F|$ shows what the plain transpose gives.)*

*Your answer.*

**2. Which columns are the 'low' and which the 'high' frequencies?**
*(Hint: compare column $k$ with column $N-k$. Which column has the fastest possible oscillation on $N$ samples?)*

*Your answer.*

---
## 8.2 DFT as a change of basis

**Task.** Compute $\hat x = F_N x$ for a random $x$ and compare with `np.fft.fft(x)`.

$$\hat x_k = \sum_{i=0}^{N-1} x_i\, e^{-j2\pi ik/N}, \qquad x_i = \frac{1}{N}\sum_{k=0}^{N-1}\hat x_k\, e^{j2\pi ki/N}.$$

### Prediction (write this *before* running)

- What scaling factor will relate $F_N x$ and `np.fft.fft(x)`?

*(fill in)*

In [ ]:
N = 16
x = np.random.default_rng(0).standard_normal(N)
F = fourier_matrix(N)

xhat_F = F @ x
xhat_np = np.fft.fft(x)
print("np.fft.fft(x) / (F_N x) =", (xhat_np / xhat_F).round(6))

xhat_sum = np.array([np.sum(x * np.exp(-2j * np.pi * np.arange(N) * k / N)) for k in range(N)])
x_back = np.array([np.sum(xhat_np * np.exp(2j * np.pi * np.arange(N) * i / N)) / N for i in range(N)])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
ax1.stem(x); ax1.set_title("x (random)"); ax1.grid(alpha=0.3)
ax2.stem(np.abs(xhat_F)); ax2.set_title("|F_N x|"); ax2.grid(alpha=0.3)
plt.show()

### Verify numerically

In [ ]:
check("direct sum formula = np.fft.fft", np.allclose(xhat_sum, xhat_np))
check("inverse sum formula recovers x", np.allclose(x_back, x))
check("np.fft.fft(x) = √N · F_N x", np.allclose(xhat_np, np.sqrt(N) * xhat_F))
check("np.fft.fft(x, norm='ortho') = F_N x", np.allclose(np.fft.fft(x, norm="ortho"), xhat_F))
check("np.fft.ifft(·, norm='ortho') = F_Nᴴ", np.allclose(np.fft.ifft(xhat_F, norm="ortho"), F.conj().T @ xhat_F))
check("F_Nᴴ (F_N x) = x  (inverse DFT [6.3.6])", np.allclose(F.conj().T @ xhat_F, x))
check("Parseval with unitary F_N: ‖F_N x‖ = ‖x‖", np.isclose(np.linalg.norm(xhat_F), np.linalg.norm(x)))
check("without normalisation: ‖fft(x)‖² = N ‖x‖²", np.isclose(np.linalg.norm(xhat_np) ** 2, N * np.linalg.norm(x) ** 2))
check("real x ⇒ x̂_(N−k) = conj(x̂_k)", np.allclose(xhat_np[1:], xhat_np[:0:-1].conj()))

---
### Explain (hand-in — in your own words)

**1. What scaling factor makes $F_N x$ and `np.fft.fft(x)` equal? Check the normalisations in [6.3.1] and the `norm="ortho"` option.**

*Your answer.*

**2. With the unitary $F_N$, verify Parseval: $\|\hat x\|_2 = \|x\|_2$. Why does a unitary matrix preserve length?**

*Your answer.*

---
## 8.3 Speed of the FFT

**Task.** Time $F_N x$ (matrix–vector) against `np.fft.fft` for $N = 2^6, \dots, 2^{14}$ and plot both on a log–log scale.

**Memory:** $F_N$ needs $16N^2$ bytes ($2^{13}$: 1.1 GB, $2^{14}$: 4.3 GB). The matrix is skipped above `MAX_MATRIX_BYTES`. Raise it only if you have the RAM. The FFT is also timed up to $2^{22}$ to see its slope more clearly.

### Prediction (write this *before* running)

- On a log–log plot, what slopes do you expect for the two methods?

*(fill in)*

In [ ]:
MAX_MATRIX_BYTES = 1.2e9

def fourier_matrix_blocks(N, block=1024):
    # same matrix as fourier_matrix, built in row blocks to limit peak memory
    k = np.arange(N)
    F = np.empty((N, N), dtype=complex)
    for start in range(0, N, block):
        kk = k[start:start + block, None]
        F[start:start + block] = np.exp(-2j * np.pi * ((kk * k) % N) / N)
    return F / np.sqrt(N)

def best_time(f, repeat):
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter(); f(); times.append(time.perf_counter() - t0)
    return min(times)

check("block construction = definition (N = 64)", np.allclose(fourier_matrix_blocks(64, block=10), fourier_matrix(64)))

rng = np.random.default_rng(0)
Ns = 2 ** np.arange(6, 15)
t_mat = []
with threadpool_limits(limits=1):          # one BLAS thread: np.fft.fft is single-threaded too, so it is a fair race
    for N in Ns:
        if 16 * N**2 > MAX_MATRIX_BYTES:
            t_mat.append(np.nan); print(f"N = {N:>6}: matrix skipped ({16 * N**2 / 1e9:.1f} GB)"); continue
        F = fourier_matrix_blocks(N)
        x = rng.standard_normal(N) + 0j
        t_mat.append(best_time(lambda: F @ x, repeat=5 if N <= 2048 else 2))
        print(f"N = {N:>6}: F_N x  {t_mat[-1] * 1e3:9.3f} ms")
        del F
t_mat = np.array(t_mat)

Ns_fft = 2 ** np.arange(6, 23)
t_fft = np.array([best_time(lambda: np.fft.fft(x, norm="ortho"), repeat=20)
                  for x in (rng.standard_normal(N) + 0j for N in Ns_fft)])

In [ ]:
def slope(n, tm, n_min):
    sel = (n >= n_min) & np.isfinite(tm)
    return np.polyfit(np.log(n[sel]), np.log(tm[sel]), 1)[0]

fig, ax = plt.subplots(figsize=(10, 6))
ax.loglog(Ns, t_mat, "o-", label=f"F_N x  (slope ≈ {slope(Ns, t_mat, 512):.2f})")
ax.loglog(Ns_fft, t_fft, "s-", label=f"np.fft.fft  (slope ≈ {slope(Ns_fft, t_fft, 2**12):.2f})")
valid = np.isfinite(t_mat)
n_ref = Ns[valid][-1]
ax.loglog(Ns, t_mat[valid][-1] * (Ns / n_ref) ** 2, "k--", lw=1, label="∝ N²")
ax.loglog(Ns_fft, t_fft[-1] * (Ns_fft * np.log2(Ns_fft)) / (Ns_fft[-1] * np.log2(Ns_fft[-1])), "k:", lw=1,
          label="∝ N log₂ N")
ax.set_xlabel("N"); ax.set_ylabel("time (s)"); ax.legend(); ax.grid(alpha=0.3, which="both")
plt.show()

---
### Explain (hand-in — in your own words)

**1. Do the slopes match the costs below [6.3.2]? matrix–vector: $O(N^2)$, FFT: $O(N\log_2 N)$.**
*(Hint: on a log–log plot, $N^p$ is a line of slope $p$. $N\log N$ has slope slightly above 1. Why are small-$N$ timings flat?)*

*Your answer.*

---
## 8.4 Circulant matrices are diagonal in the Fourier basis

**Task.** Build a random circulant matrix $C$ (`scipy.linalg.circulant`), compute $F_N C F_N^H$, and show it as a heatmap.

$$C = F_N^H \Lambda F_N, \qquad Cx = h \circledast x = F_N^H(\Lambda F_N x).$$

`circulant(h)` has $h$ as its first column; each column is the previous one shifted down by one (cyclically).

### Prediction (write this *before* running)

- What will $F_N C F_N^H$ look like? What will its diagonal be?

*(fill in)*

In [ ]:
N = 8
rng = np.random.default_rng(0)
h = rng.standard_normal(N)
C = circulant(h)
F = fourier_matrix(N)
D = F @ C @ F.conj().T

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
axes[0].imshow(C, cmap="RdBu_r"); axes[0].set_title("C = circulant(h)")
for (a, b), val in np.ndenumerate(C):
    axes[0].text(b, a, f"{val:.1f}", ha="center", va="center", fontsize=8)
im = axes[1].imshow(np.abs(D), cmap="viridis"); axes[1].set_title("|F C Fᴴ|"); fig.colorbar(im, ax=axes[1])
im = axes[2].imshow(np.log10(np.abs(D) + 1e-18), cmap="viridis"); axes[2].set_title("log₁₀ |F C Fᴴ|")
fig.colorbar(im, ax=axes[2])
plt.show()
print("largest off-diagonal |entry|:", np.abs(D - np.diag(np.diag(D))).max())

### Verify numerically

In [ ]:
Lam = np.diag(np.fft.fft(h))
x = rng.standard_normal(N)
conv = np.array([sum(h[k] * x[(i - k) % N] for k in range(N)) for i in range(N)])

check("first column of C is h", np.allclose(C[:, 0], h))
check("F C Fᴴ is diagonal", np.allclose(D - np.diag(np.diag(D)), 0))
check("diag(F C Fᴴ) = fft(h)  (DFT of the first column)", np.allclose(np.diag(D), np.fft.fft(h)))
check("C = Fᴴ Λ F", np.allclose(C, F.conj().T @ Lam @ F))
check("columns of Fᴴ are eigenvectors of C", np.allclose(C @ F.conj().T, F.conj().T @ Lam))
check("C x = h ⊛ x  (circular convolution, by the sum formula)", np.allclose(C @ x, conv))
check("C x = ifft(fft(h) · fft(x))  (FFTs + one element-wise product)",
      np.allclose(C @ x, np.fft.ifft(np.fft.fft(h) * np.fft.fft(x)).real))

---
### Explain (hand-in — in your own words)

**1. Why is $F_N C F_N^H$ diagonal?**
*(Hint: $C$ is a polynomial in the cyclic shift $S$: $C = \sum_k h_k S^k$. What does a shift do to a sampled complex exponential?)*

*Your answer.*

**2. Check that the diagonal equals the DFT of the first column $h$ of $C$, and that $Cx$ (a circular convolution) can be computed with two FFTs and one element-wise product (plus one inverse FFT). What does that cost compared with $Cx$ directly?**

*Your answer.*

---
## 8.5 Sparsity and denoising

**Task.** Reproduce Example 6.2: the signal below plus Gaussian noise. Plot the power spectrum, keep the $k$ largest coefficients, invert.

$$f(t) = \sin(2\pi\cdot 50\, t) + \sin(2\pi\cdot 120\, t) + \varepsilon(t), \qquad \varepsilon(t) \sim \mathcal{N}(0, \sigma^2).$$

As in Figure 17: sampling step $\Delta t = 0.001$ s on $[0, 1)$, so $n = 1000$ samples, and $\sigma = 2.5$. PSD $= |\hat f_k|^2 / n$.

### Prediction (write this *before* running)

- How many Fourier coefficients does the clean signal need? (Count both $\pm$ frequencies.)
- What does the spectrum of pure white noise look like?

*(fill in)*

In [ ]:
dt = 0.001
t = np.arange(0, 1, dt)
n = t.size
f_clean = np.sin(2 * np.pi * 50 * t) + np.sin(2 * np.pi * 120 * t)
freq = np.arange(n) / (n * dt)
half = np.arange(1, n // 2)

def keep_k_largest(fhat, k):
    out = np.zeros_like(fhat)
    if k > 0:
        idx = np.argsort(np.abs(fhat))[-k:]
        out[idx] = fhat[idx]
    return out

def denoise(k=4, sigma=2.5, signal="two sines", seed=0):
    rng = np.random.default_rng(seed)
    clean = f_clean if signal == "two sines" else np.zeros(n)
    f = clean + sigma * rng.standard_normal(n)
    fhat = np.fft.fft(f)
    fhat_k = keep_k_largest(fhat, k)
    f_filt = np.fft.ifft(fhat_k).real
    zoom = t < 0.25

    fig, axes = plt.subplots(3, 1, figsize=(12, 9.5))
    axes[0].plot(t[zoom], f[zoom], "r", lw=1, label="noisy")
    axes[0].plot(t[zoom], clean[zoom], "k", lw=1.5, label="clean")
    axes[1].plot(freq[half], np.abs(fhat[half]) ** 2 / n, "r", lw=1, label="noisy")
    axes[1].plot(freq[half], np.abs(fhat_k[half]) ** 2 / n, "b", lw=1.5, label=f"kept {k} largest")
    axes[2].plot(t[zoom], clean[zoom], "k", lw=1.5, label="clean")
    axes[2].plot(t[zoom], f_filt[zoom], "b", lw=1, label="filtered")
    for ax, xl in zip(axes, ["time [s]", "frequency [Hz]", "time [s]"]):
        ax.set_xlabel(xl); ax.legend(loc="upper right"); ax.grid(alpha=0.3)
    axes[1].set_ylabel("PSD")
    err = np.linalg.norm(f_filt - clean) / max(np.linalg.norm(f - clean), 1e-12)
    fig.suptitle(f"{signal}, σ = {sigma}, k = {k}:   ‖filtered − clean‖ / ‖noise‖ = {err:.3f}")
    plt.tight_layout(); plt.show()

denoise()

### Verify numerically

In [ ]:
fhat_clean = np.fft.fft(f_clean)
psd_clean = np.abs(fhat_clean) ** 2 / n
peaks = freq[half][np.argsort(psd_clean[half])[-2:]]

def energy_captured(sig, kmax=n):
    p = np.sort(np.abs(np.fft.fft(sig)) ** 2)[::-1]
    return np.cumsum(p)[:kmax] / p.sum()

check("PSD of the clean signal peaks at 50 Hz and 120 Hz", np.allclose(np.sort(peaks), [50, 120]))
check("4 coefficients hold all the energy of the clean signal (it is 4-sparse)", np.isclose(energy_captured(f_clean)[3], 1))
check("Parseval: Σ PSD = Σ f²", np.isclose(psd_clean.sum(), np.sum(f_clean**2)))
check("keeping all n coefficients gives the signal back", np.allclose(np.fft.ifft(keep_k_largest(fhat_clean, n)).real, f_clean))

### Explore: vary $k$ and $\sigma$; repeat with pure white noise

In [ ]:
@interact(k=IntSlider(min=0, max=60, step=1, value=4), sigma=FloatSlider(min=0, max=5, step=0.25, value=2.5),
          signal=Dropdown(options=["two sines", "white noise only"]), seed=IntSlider(min=0, max=20, value=0))
def explore_denoise(k, sigma, signal, seed):
    denoise(k, sigma, signal, seed)

In [ ]:
rng = np.random.default_rng(0)
curves = {"clean signal": f_clean,
          "signal + noise σ = 1": f_clean + rng.standard_normal(n),
          "signal + noise σ = 2.5": f_clean + 2.5 * rng.standard_normal(n),
          "white noise only": rng.standard_normal(n)}
fig, ax = plt.subplots(figsize=(10, 5))
kk = np.arange(1, n + 1)
for name, sig in curves.items():
    ax.semilogx(kk, energy_captured(sig), label=name)
ax.set_xlabel("k (number of largest Fourier coefficients kept)"); ax.set_ylabel("fraction of energy")
ax.legend(); ax.grid(alpha=0.3, which="both")
plt.show()

---
### Explain (hand-in — in your own words)

**1. Why is the clean signal sparse in the Fourier basis but noise is not [6.4]?**
*(Hint: how many basis vectors does one sampled sine need? White noise $\varepsilon$ is $\mathcal{N}(0, \sigma^2 I)$. What is the distribution of $F_N\varepsilon$ when $F_N$ is unitary?)*

*Your answer.*

**2. Why should $k$ be even here? What goes wrong with odd $k$?**
*(Hint: real signal ⇒ coefficients come in conjugate pairs $\hat f_k$, $\hat f_{n-k}$.)*

*Your answer.*

**3. Compare the Fourier basis (universal) with the SVD basis of the image exercise in Section 4 (tailored to the data).**
*(Hint: who chooses the basis vectors in each case, and what do you need to store or transmit along with the coefficients?)*

*Your answer.*